# Lab 2: Loss Functions and Gradient Descent

**DATA425 | Foundations of Deep Learning**

## What this lab is about

Lab 1 showed how neural networks make predictions. This lab focuses on how predictions become learning.

Training a model needs two ingredients:

1. **A loss function:** a rule that turns a prediction into a number measuring how wrong it is.
2. **An optimiser:** a rule that changes the model parameters to reduce that loss.

We will work through common losses for regression and classification, then implement gradient descent from scratch. The goal is to see that training is not magic. It is a repeated cycle of predicting, measuring error, computing a direction, and updating parameters.


## 0. Setup

Run this cell first. We use NumPy for manual calculations, Matplotlib for plots, and TensorFlow/Keras for the final comparison.

The examples are intentionally small. Small examples make it easier to see the role of each mathematical object before we apply the same ideas to large neural networks.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "tensorflow": "tensorflow"
}

missing = [package for module, package in REQUIRED.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Automatic installation failed. Install these packages manually or run this notebook "
            "in an environment with internet access: " + ", ".join(missing)
        ) from exc
else:
    print("All required packages are already installed.")

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    pass

SEED = 425
rng = np.random.default_rng(SEED)
tf.keras.utils.set_random_seed(SEED)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

The setup defines a random seed and plotting defaults. The seed helps make the generated examples stable from run to run.


## 1. Regression losses

In regression, the target is a number. The model might predict a house price, a temperature, a demand forecast, or any other continuous value.

The prediction error is:

$$\text{error} = \hat y - y.$$

Two common regression losses are:

- **Mean Absolute Error (MAE):** average size of the errors, ignoring direction.
- **Mean Squared Error (MSE):** average squared error.

Squaring makes large errors much more expensive. That can be useful when large mistakes are especially bad, but it also means MSE is more sensitive to outliers than MAE.


In [ ]:
y_true = np.array([80, 20, 70, 65, 95], dtype=float)
y_pred = np.array([76,  2, 82, 66, 92], dtype=float)

errors = y_pred - y_true
absolute_errors = np.abs(errors)
squared_errors = errors ** 2

mae = np.mean(absolute_errors)
mse = np.mean(squared_errors)
rmse = np.sqrt(mse)

print("Prediction errors:", errors)
print(f"MAE : {mae:.2f}")
print(f"MSE : {mse:.2f}")
print(f"RMSE: {rmse:.2f}")

x = np.arange(len(y_true))
plt.bar(x - 0.2, y_true, width=0.4, label="true")
plt.bar(x + 0.2, y_pred, width=0.4, label="predicted")
plt.xticks(x, [f"case {i+1}" for i in x])
plt.ylabel("value")
plt.title("True values and predictions")
plt.legend()
plt.show()

Notice how one large error affects MSE more strongly than MAE. This is why choosing a loss function is part of the modelling decision. The loss tells the model what kind of mistake should matter most.


## 2. Binary cross-entropy

For binary classification, the model predicts a probability such as $P(y=1)=0.8$.

Binary cross-entropy rewards the model for assigning high probability to the correct class. It strongly punishes confident wrong predictions. This is exactly what we usually want from a classifier: being wrong is bad, and being very confident while wrong is worse.

For one example, the binary cross-entropy loss is:

$$L = -\left[y\log(p) + (1-y)\log(1-p)\right].$$

Here, $p$ is the predicted probability of class 1.


In [ ]:
def binary_cross_entropy(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

cases = [
    (1, 0.95, "correct and confident"),
    (1, 0.55, "correct but unsure"),
    (1, 0.05, "wrong and confident"),
    (0, 0.05, "correct and confident"),
    (0, 0.45, "correct but unsure"),
    (0, 0.95, "wrong and confident"),
]

for y, p, description in cases:
    print(f"y={y}, predicted P(y=1)={p:.2f}, loss={binary_cross_entropy(y, p):.3f}, {description}")

prob_grid = np.linspace(0.001, 0.999, 300)
plt.plot(prob_grid, binary_cross_entropy(1, prob_grid), label="true class is 1")
plt.plot(prob_grid, binary_cross_entropy(0, prob_grid), label="true class is 0")
plt.xlabel("predicted P(y=1)")
plt.ylabel("binary cross-entropy")
plt.title("Confident wrong predictions are expensive")
plt.legend()
plt.show()

Look at the curve for the true class. The loss is small when the model assigns high probability to the correct answer. The loss grows quickly when the model assigns low probability to the correct answer.


## 3. Softmax and multi-class cross-entropy

For multi-class classification, the model produces one score for each class. These raw scores are often called **logits**.

Softmax converts logits into probabilities:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}.$$

The probabilities add to 1, so we can read them as the model's belief across the possible classes.

Multi-class cross-entropy then looks only at the probability assigned to the true class:

$$L = -\log(p_{\text{true class}}).$$

If the model gives the correct class high probability, the loss is low. If it gives the correct class low probability, the loss is high.


In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)

logits = np.array([
    [2.0, 1.0, 0.2],
    [0.1, 3.0, 0.4],
    [2.5, 2.4, 2.3],
])
true_classes = np.array([0, 1, 2])
probs = softmax(logits)
losses = -np.log(probs[np.arange(len(true_classes)), true_classes])

print("Softmax probabilities:")
print(np.round(probs, 3))
print("Per-example cross-entropy:", np.round(losses, 3))
print("Mean cross-entropy:", f"{losses.mean():.3f}")

The softmax function is shifted in the code before exponentiating. This does not change the probabilities, but it improves numerical stability by avoiding unnecessarily huge exponentials.


## 4. Gradient descent from scratch

A loss function tells us how wrong the model is. Gradient descent tells us how to change a parameter to make the loss smaller.

For one parameter $w$, the update rule is:

$$w \leftarrow w - \eta \frac{dL}{dw}.$$

The derivative tells us the local slope of the loss. The learning rate $\eta$ controls the step size.

- If the learning rate is too small, training can be slow.
- If the learning rate is too large, training can jump around or diverge.
- If the learning rate is sensible, the parameter moves steadily toward a lower loss.


In [ ]:
def loss(w):
    return (w - 3) ** 2 + 1

def grad(w):
    return 2 * (w - 3)

def run_gradient_descent(start, learning_rate, steps=30):
    ws = [start]
    for _ in range(steps):
        ws.append(ws[-1] - learning_rate * grad(ws[-1]))
    return np.array(ws)

learning_rates = [0.05, 0.3, 0.9]
w_values = np.linspace(-2, 6, 400)
plt.plot(w_values, loss(w_values), color="black", label="loss curve")

for lr in learning_rates:
    path = run_gradient_descent(start=-1.5, learning_rate=lr)
    plt.plot(path, loss(path), marker="o", markersize=3, label=f"learning rate {lr}")

plt.xlabel("w")
plt.ylabel("loss")
plt.title("Gradient descent depends on the learning rate")
plt.legend()
plt.show()

Compare the paths for the different learning rates. The gradient gives a direction, but the learning rate determines how far we move in that direction.


## 5. Linear regression trained by gradient descent

Now we train a tiny linear model from scratch:

$$\hat y = wx + b.$$

This example has only two trainable parameters: the slope $w$ and the intercept $b$. In a neural network, there may be thousands or millions of weights, but the same idea applies.

Each epoch does the following:

1. Make predictions with the current values of $w$ and $b$.
2. Compute the MSE.
3. Compute the gradients of the MSE with respect to $w$ and $b$.
4. Move $w$ and $b$ in the direction that reduces the loss.


In [ ]:
rng = np.random.default_rng(SEED)
X = rng.uniform(-3, 3, size=120)
y = 2.5 * X - 1.0 + rng.normal(0, 0.8, size=X.shape)

w, b = 0.0, 0.0
learning_rate = 0.05
history = []

for epoch in range(120):
    y_hat = w * X + b
    residuals = y_hat - y
    mse = np.mean(residuals ** 2)
    history.append(mse)

    dw = 2 * np.mean(residuals * X)
    db = 2 * np.mean(residuals)

    w -= learning_rate * dw
    b -= learning_rate * db

print(f"Learned w: {w:.3f}")
print(f"Learned b: {b:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("MSE")
axes[0].set_title("Loss decreases during training")

axes[1].scatter(X, y, alpha=0.7, edgecolors="k", linewidth=0.3)
line_x = np.linspace(X.min(), X.max(), 100)
axes[1].plot(line_x, w * line_x + b, color="black", linewidth=2)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Fitted linear model")
plt.tight_layout()
plt.show()

The loss curve should go down over time. The fitted line should follow the overall trend of the data, even though the data contains noise. This is what learning looks like in the simplest possible regression model.


## 6. The same idea in Keras

Keras performs the same training loop for us:

1. Run a forward pass.
2. Compute the loss.
3. Use automatic differentiation to compute gradients.
4. Use the optimiser to update the weights.

The main difference is that Keras can do this for much larger models without us writing out every derivative by hand.

In this cell, the model has one dense layer with one output. That is enough to represent the same line $\hat y = wx + b$.


In [ ]:
tf.keras.utils.set_random_seed(SEED)

model = keras.Sequential([
    keras.Input(shape=(1,)),
    layers.Dense(1),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.05),
    loss="mse",
)
history_keras = model.fit(X.reshape(-1, 1), y, epochs=25, batch_size=16, verbose=0)

w_keras, b_keras = model.layers[0].get_weights()
print(f"Keras learned w: {w_keras[0, 0]:.3f}")
print(f"Keras learned b: {b_keras[0]:.3f}")

plt.plot(history_keras.history["loss"])
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.title("Keras training curve")
plt.show()

The learned Keras slope and intercept should be close to the values learned from the manual gradient descent code. This is a useful sanity check: Keras automates the training process, but it is still optimising a loss with gradients.


## Summary

In this lab, you connected the core pieces of training:

- Loss functions measure how wrong a prediction is.
- MAE and MSE are common regression losses, with MSE punishing large errors more strongly.
- Binary cross-entropy is common for binary classification.
- Softmax plus cross-entropy is common for multi-class classification.
- Gradient descent uses derivatives to update parameters.
- Keras automates the same training loop that we can write by hand for small models.

The key idea is that learning is driven by the loss. When you choose a loss function, you choose what kind of error the model is trained to avoid.
